# ASTER block 2 — training run

Runs the whole block-2 pipeline: corpus → cell head → MIL head → threshold resolution →
frozen-grid evaluation on cAItomorph (409 patients) → ×40 stress test → ONNX export.

**Before running anything**, read `PREREGISTRATION.md`. Two rules govern this notebook:

1. **cAItomorph is never trained on and never fitted on.** It is only read in §9 and §10.
2. Values tagged **[REF]** and **[FIT]** are resolved in §7 from development and control
   data only, then written back to `decision_grid.yaml` as a dated amendment.

Every section is idempotent: it skips its work if the artifact already exists in Drive.

Runtime: **T4 is enough** (A100 halves §4 and §6). Expect ≈ 3–5 h end to end on a T4.

## 0. Environment

In [ ]:
import os, sys, json, time, random, hashlib, subprocess
from pathlib import Path

SEED = 42
random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

import torch, numpy as np
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("torch:", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Everything durable lives in Drive so a disconnect never costs a training run.
WORK  = Path("/content/drive/MyDrive/aster_block2")      # checkpoints, results, logs
DATA  = Path("/content/data")                             # corpus, local SSD, fast
REPO  = Path("/content/aster-block2")                     # this repository
for path in (WORK, DATA, WORK/"checkpoints", WORK/"results", WORK/"logs"):
    path.mkdir(parents=True, exist_ok=True)
print(WORK, DATA, REPO)

### 0.1 Upload the repository

Either upload `aster-block2.zip` (produced on the Mac by `data_bridge/pack.py --repo-only`)
to Drive, or clone it if you have pushed it to git. The zip route needs no remote.

In [ ]:
# --- bridge credentials, then bootstrap the repository ---------------------------------
# Paste the two values printed by `bash data_bridge/serve.sh` on the Mac.
BRIDGE_URL   = ""          # e.g. "https://xxxx.trycloudflare.com"
BRIDGE_TOKEN = ""

# Chicken and egg: the repo's own downloader lives inside the repo, and the repo arrives
# over the bridge. This bootstrap fetches the `payload` dataset - the repository zip and
# the block-1 weights - with the standard library alone; everything after it uses the
# repo's tooling.
import io, tarfile, urllib.request, zipfile

def _bridge(path):
    request = urllib.request.Request(f"{BRIDGE_URL.rstrip('/')}/{path}",
                                     headers={"Authorization": f"Bearer {BRIDGE_TOKEN}"})
    return urllib.request.urlopen(request, timeout=300)

PAYLOAD = DATA/"payload"
if not (PAYLOAD/"aster-block2.zip").exists():
    assert BRIDGE_URL and BRIDGE_TOKEN, "paste BRIDGE_URL and BRIDGE_TOKEN above"
    PAYLOAD.mkdir(parents=True, exist_ok=True)
    plan = json.load(_bridge("shards/payload.json"))
    for shard in plan["shards"]:
        blob = _bridge(f"tar/payload/{shard['id']}").read()
        with tarfile.open(fileobj=io.BytesIO(blob)) as archive:
            archive.extractall(PAYLOAD)
    print("payload:", sorted(p.name for p in PAYLOAD.iterdir()))
if not (REPO/"data_bridge/colab_pull.py").exists():
    with zipfile.ZipFile(PAYLOAD/"aster-block2.zip") as archive:
        archive.extractall(REPO)
assert (REPO/"data_bridge/colab_pull.py").exists(), "the repository zip is incomplete"

WEIGHTS = PAYLOAD/"wbc_detector.pt"          # used by section 2 for the x40 crops
sys.path.insert(0, str(REPO/"src"))
sys.path.insert(0, str(REPO/"datasets"))

# Install only what the Colab image lacks. `|| true`: a pip failure must never abort an
# unattended run, and the image ships its own torch/numpy - pinning against it makes pip
# refuse the whole set.
!pip -q install -r {REPO}/env/requirements-colab.txt || true
import importlib
for _module in ("sklearn", "scipy", "pandas", "yaml", "onnxruntime", "ultralytics"):
    try:
        importlib.import_module(_module)
    except ImportError:
        print(f"  MISSING: {_module}")

from aster_block2.grid import Thresholds, evaluate, load_thresholds
from aster_block2.proportion_test import test_proportion, min_count_to_assert
from aster_block2.models import Encoder, CellHead, GatedAttentionMIL, partial_label_loss, attention_entropy_penalty, encode_bag
from aster_block2.preprocess import build_eval_transform, IMAGENET_MEAN, IMAGENET_STD
from aster_block2.schema import SessionResult
print("aster_block2 imported from", REPO)

In [ ]:
# --- the run directory: everything this run leaves behind, in Drive, as it happens -----
from aster_block2.runlog import RunDir

run = RunDir.create(WORK, repo=REPO)
run.install_exception_hook()   # any cell error lands in ALERTS.md, even unattended

# The run writes ~0.4 GB to Drive (weights, ONNX, logs, results). A full Drive makes the
# FIRST checkpoint fail hours into training - better to know now.
import shutil as _shutil
_drive_free = _shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Drive: {_drive_free:.2f} GB free")
assert _drive_free > 0.6, (f"only {_drive_free:.2f} GB free in Drive; the run needs ~0.4 GB "
                           f"for weights and results. Free some space and re-run.")
print(f"""
Everything is written to {run.root} as it happens:
  logs/        stdout of every section + traceback if it fails
  metrics/     one JSON line per epoch -> the curves survive a disconnection
  checkpoints/ per epoch, the last 2 kept + <name>_final.pt
  results/     every CSV/JSON that the article cites
  bundle/      exactly what integration/PATCH.md expects on the Jetson
  manifest.json + env/  versions, GPU, seeds, pre-registration digests
A disconnection therefore never costs more than the current epoch.
""")

In [ ]:
# The frozen grid must be the frozen grid. This aborts if anything drifted.
expected = {}
for line in (REPO/"results/PREREGISTRATION.sha256").read_text().splitlines():
    if line.startswith("#") or not line.strip(): continue
    digest, name = line.split(None, 1)
    expected[name.strip()] = digest

for name, digest in expected.items():
    actual = hashlib.sha256((REPO/name).read_bytes()).hexdigest()
    status = "OK" if actual == digest else "CHANGED"
    print(f"  {status:8} {name}")
    assert actual == digest, f"{name} differs from the frozen pre-registration"
GRID_SHA = expected["src/aster_block2/decision_grid.yaml"]
print("\ngrid sha256:", GRID_SHA)

## 1. Corpus

The Mac produced `_work/corpus/` — every image square-padded and stored lossless WebP,
plus `manifest.csv` and `splits.csv`. Pull it here with the bridge (§1a) or from
Drive (§1b). ~8 GB.

In [ ]:
# --- 1. the corpus and the x40 fields ---------------------------------------------------
# In-process, not subprocess: Colab's sys.executable is a different interpreter from the
# kernel and a child's stderr is swallowed. No existence guard: manifest.csv arrives in ONE
# shard, so its presence does not mean the corpus is complete; pull_dataset is itself
# idempotent and resumes shard by shard.
sys.path.insert(0, str(REPO/"data_bridge"))
import colab_pull

for _name in ("corpus", "x40_fields"):
    colab_pull.pull_dataset(BRIDGE_URL, BRIDGE_TOKEN, _name, DATA, workers=4)

In [ ]:
# --- the manifest, merged with the splits ---------------------------------------------
import pandas as pd
manifest = pd.read_csv(DATA/"corpus/manifest.csv")
splits   = pd.read_csv(DATA/"corpus/splits.csv")
manifest = manifest.merge(splits[["corpus_path","split"]], on="corpus_path", how="left")
print(manifest.groupby(["source","split"]).size().to_string())
print("\ntotal images:", len(manifest))

## 2. Prototype ×40 crops (block 1)

Runs the **deployed** localizer on the ×40 fields to regenerate the crops the deployed
pipeline would produce. Same weights, same imgsz 960 / conf 0.18 / iou 0.50 / padding 0.10
as `config/inference.yaml`, so the crops are contract-identical.

These crops are unlabeled, non-leukemic material. They are the **OOD stress set** — never
supervised training data for any leukemia class.

In [ ]:
# Completion is a MARKER, not the existence of the output directory: the directory is
# created before the loop, so a crash mid-way used to leave a partial crop set that the
# next run reported as "already present" - an incomplete stress test, silently.
import shutil
X40_OUT = DATA/"x40_sessions"
DONE = X40_OUT/".complete"
if not DONE.exists():
    shutil.rmtree(X40_OUT, ignore_errors=True)        # discard any partial result
    !pip -q install ultralytics
    from ultralytics import YOLO
    from PIL import Image
    from aster_block2.crops import crops_from_boxes   # the deployed crop convention

    assert WEIGHTS.exists(), "wbc_detector.pt missing from the bridge payload"
    detector = YOLO(str(WEIGHTS), task="detect")

    fields = sorted(p for p in (DATA/"x40_fields").rglob("*")
                    if p.suffix.lower() in {".jpg", ".png", ".jpeg"})
    print(len(fields), "x40 fields")

    CONF, IOU, IMGSZ = 0.18, 0.50, 960        # config/inference.yaml
    crop_dir = X40_OUT/"x40_all"/"crops"; crop_dir.mkdir(parents=True, exist_ok=True)
    kept, unreadable = 0, []
    for index, field in enumerate(fields):
        # One source field (train/slide_20260908_162808_658.jpg) has a broken JPEG stream.
        # OpenCV - what YOLO reads with - tolerates it; Pillow - what the crop is cut with -
        # does not. An unreadable field is excluded and named, never half-decoded.
        try:
            with Image.open(field) as _image:
                _image.convert("RGB")
        except OSError as exc:
            unreadable.append(f"{field.relative_to(DATA/'x40_fields')}: {exc}")
            continue
        boxes = [b.xyxy[0].tolist() for b in detector.predict(
            str(field), imgsz=IMGSZ, conf=CONF, iou=IOU, classes=[0], verbose=False)[0].boxes]
        # crops_from_boxes applies expand_and_clip_box + extract_crop exactly as the
        # deployed pipeline does: +10%/side, clipped, native pixels, and a degenerate box
        # drops the detection entirely rather than yielding a padded stub.
        for _, crop in crops_from_boxes(field, boxes):
            crop.save(crop_dir/f"field{index:03d}_wbc_{kept:04d}.png")
            kept += 1
    if unreadable:
        run.alert(f"{len(unreadable)} x40 field(s) unreadable, excluded from the stress test",
                  detail="\n".join(unreadable))
    usable = len(fields) - len(unreadable)
    DONE.write_text(f"{kept} crops from {usable} fields, {len(unreadable)} unreadable\n")
    print(f"x40 crops: {kept} from {usable} fields ({kept/max(usable,1):.2f} WBC/field), "
          f"{len(unreadable)} unreadable")
else:
    print("x40 crops already complete:", DONE.read_text().strip())

## 3. Datasets and loaders

`stored_side` in the manifest is ≤ 256; the loader resizes to 224, which is the deployed
contract. Training augmentation is the stain/scale-invariance set: strong HSV jitter (the
block-1 recipe used `hsv_h=0.5`), blur↔sharpen as a magnification proxy, small rotations,
JPEG artifacts.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

CELL_CLASSES = ["myeloblast","lymphoblast","promyelocyte","promyelocyte_abnormal",
                "myelocyte","metamyelocyte","band_neutrophil","segmented_neutrophil",
                "basophil","eosinophil","monocyte","lymphocyte","lymphocyte_atypical",
                "smudge_cell","erythroblast","other_artifact"]
CLASS_INDEX = {name: i for i, name in enumerate(CELL_CLASSES)}

train_tf = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(20)], p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.4, hue=0.25),  # stain variation
    T.RandomApply([T.GaussianBlur(5, (0.1, 2.5))], p=0.4),                  # magnification proxy
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.15, scale=(0.02, 0.08)),
])
eval_tf = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class CellDataset(Dataset):
    """Cells with a definite OR partial label. `allowed` is a 0/1 mask over classes."""
    def __init__(self, frame, root, transform):
        self.rows = frame.reset_index(drop=True); self.root = Path(root); self.tf = transform
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        image = Image.open(self.root/row.corpus_path).convert("RGB")
        allowed = torch.zeros(len(CELL_CLASSES))
        for name in str(row.cell_labels).split("|"):
            if name in CLASS_INDEX: allowed[CLASS_INDEX[name]] = 1.0
        return self.tf(image), allowed, torch.tensor(float(row.weight))

cell_rows = manifest[manifest.cell_labels.notna() & (manifest.cell_labels != "")]
train_rows = cell_rows[cell_rows.split == "cell_train"]
val_rows   = cell_rows[cell_rows.split == "cell_val"]
print(f"cell head: {len(train_rows)} train / {len(val_rows)} val")
print(train_rows.cell_labels.value_counts().head(20).to_string())

## 4. Cell head

Encoder + 16-way head, trained with the partial-label loss so that a cell annotated only
as "not a blast" (ALL-IDB non-cancer) or "one of the immature granulocytes" (PBC `ig`) is
never forced into a class its annotation does not support.

Class imbalance is extreme by design of the sources: `segmented_neutrophil` 8484 against
`lymphocyte_atypical` 11. Sampling is class-balanced; per-class recall is reported
separately in §5 and carried into the paper, because the four rarest classes are exactly
the ones the chronic rules depend on.

In [ ]:
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import f1_score, recall_score
from aster_block2.quantify import quantification_error

CKPT = WORK/"checkpoints/cell_head.pt"
EPOCHS_CELL, PATIENCE_CELL = 60, 8

# Monitored metric: MACRO F1 over the definite-label validation cells. Not macro RECALL,
# which was the first choice and was wrong: it rewards OVER-predicting rare classes, and
# over-predicting smudge cells is exactly what makes R3 fire on normal blood. The grid
# consumes PROPORTIONS, so precision matters as much as recall. The quantification MAE is
# logged alongside as the diagnostic closest to what the grid actually reads.
# (kept for the record) macro recall over the definite-label validation cells - the unweighted
# mean over classes. Overall accuracy would be dominated by segmented_neutrophil (15 253
# cells) and would happily ignore smudge_cell, lymphocyte_atypical and
# promyelocyte_abnormal - precisely the classes the chronic rules and the APL flag depend
# on. Macro recall makes the rare classes count as much as the common ones.

def make_balanced_sampler(frame):
    counts = frame.cell_labels.value_counts()
    weights = frame.cell_labels.map(lambda k: 1.0/counts[k]).to_numpy()
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(frame), replacement=True)

encoder = Encoder(pretrained=True).to(DEVICE)
cell_head = CellHead().to(DEVICE)

if CKPT.exists():
    state = torch.load(CKPT, map_location=DEVICE)
    encoder.load_state_dict(state["encoder"]); cell_head.load_state_dict(state["cell_head"])
    print("loaded", CKPT)
else:
    train_loader = DataLoader(CellDataset(train_rows, DATA/"corpus", train_tf), batch_size=64,
                              sampler=make_balanced_sampler(train_rows), num_workers=2, drop_last=True)
    val_loader   = DataLoader(CellDataset(val_rows, DATA/"corpus", eval_tf), batch_size=128, num_workers=2)
    optimizer = torch.optim.AdamW(list(encoder.parameters())+list(cell_head.parameters()), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS_CELL)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE=="cuda")

    best, best_epoch, best_state, waited = -1.0, 0, None, 0
    for epoch in range(1, EPOCHS_CELL+1):
        encoder.train(); cell_head.train(); running = 0.0
        for images, allowed, weights in train_loader:
            images, allowed, weights = images.to(DEVICE), allowed.to(DEVICE), weights.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=DEVICE=="cuda"):
                logits = cell_head(encoder(images))["logits"]
                log_probabilities = torch.log_softmax(logits, -1)
                mass = torch.logsumexp(log_probabilities.masked_fill(allowed==0, -1e30), -1)
                loss = -(mass * weights).mean()
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            running += loss.item()
        scheduler.step()

        encoder.eval(); cell_head.eval()
        preds, truth, in_allowed, total = [], [], 0, 0
        with torch.inference_mode():
            for images, allowed, _ in val_loader:
                logits = cell_head(encoder(images.to(DEVICE)))["logits"].cpu()
                p = logits.argmax(-1)
                in_allowed += allowed[torch.arange(len(p)), p].sum().item(); total += len(p)
                definite = allowed.sum(1) == 1
                if definite.any():
                    preds += p[definite].tolist(); truth += allowed[definite].argmax(-1).tolist()
        macro = f1_score(truth, preds, average="macro", zero_division=0) if truth else 0.0
        macro_recall = recall_score(truth, preds, average="macro", zero_division=0) if truth else 0.0
        from collections import Counter as _C
        qmae = quantification_error(_C(truth), _C(preds)) if truth else 1.0
        print(f"epoch {epoch:3d}  loss {running/len(train_loader):.4f}  "
              f"macro-F1 {macro:.4f}  macro-recall {macro_recall:.4f}  quant-MAE {qmae:.4f}"
              + ("   <- best" if macro > best else f"   (patience {waited+1}/{PATIENCE_CELL})"))
        run.metric("cell_head", epoch=epoch, loss=running/len(train_loader),
                   val_macro_f1=macro, val_macro_recall=macro_recall,
                   val_quantification_mae=qmae, val_in_allowed_set=in_allowed/total,
                   lr=scheduler.get_last_lr()[0])

        if macro > best:
            best, best_epoch, waited = macro, epoch, 0
            best_state = {"encoder": {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()},
                          "cell_head": {k: v.detach().cpu().clone() for k, v in cell_head.state_dict().items()}}
            run.checkpoint("cell_head", best_state, epoch=epoch, keep_last=1)
        else:
            waited += 1
            if waited >= PATIENCE_CELL:
                print(f"early stop at epoch {epoch}; best was epoch {best_epoch} "
                      f"(macro-F1 {best:.4f})")
                break

    encoder.load_state_dict(best_state["encoder"]); cell_head.load_state_dict(best_state["cell_head"])
    encoder.to(DEVICE); cell_head.to(DEVICE)
    torch.save(best_state, CKPT)
    run.result("cell_head_training.json",
               {"best_epoch": best_epoch, "best_macro_f1": best, "epochs_run": epoch,
                "max_epochs": EPOCHS_CELL, "patience": PATIENCE_CELL,
                "monitor": "macro F1 on definite-label cell_val"})
    print("saved", CKPT, f"(best epoch {best_epoch})")

## 5. Per-class quality, and the differential validated against AML-MLL

§6.1 and §6.2 of the pre-registration. Two things are measured here and both go into the
paper:

* **per-class precision/recall** — the chronic rules rest on `smudge_cell`,
  `lymphocyte_atypical`, `metamyelocyte`, `myelocyte`, the four rarest classes;
* **the bag differential against the manual differential** — AML-MLL carries `pb_*`
  columns, a manual per-patient count in percent. Spearman ρ **per class**, never a
  single average.

`ρ < 0.5` on `myeloblast` is a declared falsification criterion (§6.5): the grid would be
withdrawn and only `P_abn` reported.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from scipy.stats import spearmanr

# --- per-class report on definite-label validation cells ---------------------
definite = val_rows[~val_rows.cell_labels.str.contains(r"\|", na=False)]
loader = DataLoader(CellDataset(definite, DATA/"corpus", eval_tf), batch_size=128, num_workers=2)
predictions, truths = [], []
encoder.eval(); cell_head.eval()
with torch.inference_mode():
    for images, allowed, _ in loader:
        predictions += cell_head(encoder(images.to(DEVICE)))["logits"].argmax(-1).cpu().tolist()
        truths += allowed.argmax(-1).tolist()
report = classification_report(truths, predictions, labels=list(range(len(CELL_CLASSES))),
                               target_names=CELL_CLASSES, output_dict=True, zero_division=0)
import pandas as pd
report_frame = pd.DataFrame(report).T
print(report_frame.to_string())
report_frame.to_csv(WORK/"results/cell_head_per_class.csv")
run.result("cell_head_per_class.csv", report_frame.reset_index())

In [ ]:
# --- the quantifier: what turns cell predictions into defensible PROPORTIONS ----------
# Fitted on the same definite-label validation cells, per GRID QUANTITY - blast_frac is a
# sum of three classes, so its error rate must be measured on that sum. This is what stops
# R3 from firing on a normal smear because 5 % of leukocytes were miscalled smudge cells.
from aster_block2.quantify import Quantifier
from aster_block2.grid import QUANTITIES

true_names = [CELL_CLASSES[i] for i in truths]
pred_names = [CELL_CLASSES[i] for i in predictions]
quantifier = Quantifier.fit_groups(true_names, pred_names, QUANTITIES)
quantifier.save(WORK/"checkpoints/quantifier.json")
run.result("quantifier.json", quantifier.report())

print(f"{'quantity':<18}{'TPR':>8}{'FPR':>8}{'sep':>8}   quantifiable")
for name, r in quantifier.report().items():
    flag = "yes" if r["quantifiable"] else "NO  <- the grid must abstain on this criterion"
    print(f"{name:<18}{r['tpr']:>8.3f}{r['fpr']:>8.3f}{r['separation']:>8.3f}   {flag}")
unquantifiable = [k for k, v in quantifier.report().items() if not v["quantifiable"]]
if unquantifiable:
    run.alert(f"{unquantifiable} cannot support a proportion claim (TPR - FPR < 0.05). "
              f"Any rule resting on them returns indeterminate by construction - report "
              f"it, do not retune.")

In [ ]:
# --- bag differential vs the AML-MLL manual differential ---------------------
PB_MAP = {  # manual differential column -> our classes
    "pb_myeloblast": ["myeloblast"], "pb_promyelocyte": ["promyelocyte","promyelocyte_abnormal"],
    "pb_myelocyte": ["myelocyte"], "pb_metamyelocyte": ["metamyelocyte"],
    "pb_neutrophil_band": ["band_neutrophil"], "pb_neutrophil_segmented": ["segmented_neutrophil"],
    "pb_eosinophil": ["eosinophil"], "pb_basophil": ["basophil"], "pb_monocyte": ["monocyte"],
    "pb_lymph_typ": ["lymphocyte"], "pb_lymph_atyp_react": ["lymphocyte_atypical"],
}

def predict_counts(paths, batch=128):
    counts = {name: 0 for name in CELL_CLASSES}
    for start in range(0, len(paths), batch):
        images = torch.stack([eval_tf(Image.open(DATA/"corpus"/p).convert("RGB"))
                              for p in paths[start:start+batch]]).to(DEVICE)
        with torch.inference_mode():
            for index in cell_head(encoder(images))["logits"].argmax(-1).cpu().tolist():
                counts[CELL_CLASSES[index]] += 1
    return counts

mll = manifest[manifest.source == "aml_mll"].copy()
mll["differential"] = mll.extra.map(lambda s: json.loads(s).get("differential", {}))
rows = []
for patient, group in mll.groupby("patient_id"):
    manual = group.differential.iloc[0]
    if not manual or not manual.get("pb_total"): continue
    counts = predict_counts(group.corpus_path.tolist())
    total = sum(counts[c] for c in CELL_CLASSES)
    if not total: continue
    record = {"patient_id": patient, "n_cells": total}
    for column, classes in PB_MAP.items():
        record[f"manual_{column}"] = float(manual.get(column) or 0)
        record[f"pred_{column}"] = 100.0 * sum(counts[c] for c in classes) / total
    rows.append(record)
differential_frame = pd.DataFrame(rows)
differential_frame.to_csv(WORK/"results/differential_validation.csv", index=False)

print(f"{'column':<28}{'rho':>8}{'p':>10}   n={len(differential_frame)}")
correlations = {}
for column in PB_MAP:
    rho, pvalue = spearmanr(differential_frame[f"manual_{column}"], differential_frame[f"pred_{column}"])
    correlations[column] = rho
    print(f"{column:<28}{rho:>8.3f}{pvalue:>10.2g}")
json.dump(correlations, open(WORK/"results/differential_spearman.json","w"), indent=2)
run.result("differential_spearman.json", correlations)
run.result("differential_validation.csv", differential_frame)

if correlations["pb_myeloblast"] < 0.5:
    run.alert(f"criterion 6.5 MET: rho(myeloblast) = {correlations['pb_myeloblast']:.3f} < 0.5. "
              f"The grid is withdrawn; report P_abn only. Do not retune.",
              severity="FALSIFIER")

## 6. Attention-MIL head

Trained on `train_mil` patients only (AML-MLL), on **cached embeddings** — the encoder is
frozen here, so one pass over the corpus feeds every epoch. `P_abn` is what R1 and R4 use;
it is recorded but never used to assert a chronic pattern (§4.1), because it was trained on
AML versus control and has never seen a chronic entity.

In [ ]:
# 166 MB and recomputable in minutes from the encoder: local disk, not Drive.
FEATURES = DATA/"features_mll.pt"
if FEATURES.exists():
    cache = torch.load(FEATURES)
else:
    cache = {}
    encoder.eval()
    for patient, group in mll.groupby("patient_id"):
        paths = group.corpus_path.tolist(); embeddings = []
        for start in range(0, len(paths), 128):
            images = torch.stack([eval_tf(Image.open(DATA/"corpus"/p).convert("RGB"))
                                  for p in paths[start:start+128]]).to(DEVICE)
            with torch.inference_mode():
                embeddings.append(encoder(images).cpu())
        cache[patient] = torch.cat(embeddings)
    torch.save(cache, FEATURES)
print(len(cache), "patients cached,", sum(v.shape[0] for v in cache.values()), "cells")

patient_split = dict(zip(splits[splits.source=="aml_mll"].patient_id, splits[splits.source=="aml_mll"].split))
patient_label = {p: g.bag_label.iloc[0] for p, g in mll.groupby("patient_id")}
is_positive  = {p: 0 if label == "control" else 1 for p, label in patient_label.items()}
train_patients = [p for p in cache if patient_split.get(p) == "train_mil"]
fit_patients   = [p for p in cache if patient_split.get(p) == "dev_fit"]
cal_patients   = [p for p in cache if patient_split.get(p) == "dev_cal"]
print(f"MIL: {len(train_patients)} train / {len(fit_patients)} dev_fit / {len(cal_patients)} dev_cal")

In [ ]:
from sklearn.metrics import roc_auc_score

from aster_block2.sampling import deterministic_bag

def p_abn_raw(patient):
    """Raw MIL probability on the bag SessionScorer draws at inference.

    Every evaluation of P_abn - early stopping, tau_abn, temperature - uses this, so the
    scores a threshold is fitted on are the scores it will be compared with. Taking
    cache[patient][:BAG] used the first 200 cells in file order instead of the seeded
    random bag of aster_pipeline/sampling.py.
    """
    feats = cache[patient]
    with torch.inference_mode():
        return mil(feats[deterministic_bag(len(feats), BAG, SEED)].to(DEVICE))["probability"].item()

MIL_CKPT = WORK/"checkpoints/mil_head.pt"
BAG, EPOCHS_MIL, PATIENCE_MIL = 200, 300, 30

# Early stopping is model selection, so it is paid for out of the TRAINING budget:
# mil_val patients, carved from train_mil by build_splits.py. Stopping on dev_fit would
# choose the epoch on the same patients that later fix tau_abn and theta_APL, and those
# operating points would look better than they are.
val_patients = [p for p in cache if patient_split.get(p) == "mil_val"]
print(f"early stopping on {len(val_patients)} mil_val patients "
      f"({sum(is_positive[p] for p in val_patients)} positive)")

mil = GatedAttentionMIL().to(DEVICE)
if MIL_CKPT.exists():
    mil.load_state_dict(torch.load(MIL_CKPT, map_location=DEVICE)); print("loaded", MIL_CKPT)
else:
    optimizer = torch.optim.AdamW(mil.parameters(), lr=1e-4, weight_decay=1e-4)
    generator = random.Random(SEED)
    positives = [p for p in train_patients if is_positive[p]]
    negatives = [p for p in train_patients if not is_positive[p]]

    def evaluate_on(patients):
        """AUROC (the declared monitor, amendment 5a) and mean BCE (logged, diagnostic only).

        Run 20260911_051357Z: AUROC reached 1.0 on the 29 mil_val patients at epoch 1, so
        the declared strict-improvement rule kept epoch 1 (train BCE 0.29) and stopped at
        31. Selection is left AUROC-only so this notebook reproduces the reported run; a
        monitor that does not saturate is recommended for the next version, as a declared
        change (results/RUN_NOTES_20260911.md).
        """
        mil.eval(); scores, labels = [], []
        with torch.inference_mode():
            for patient in patients:
                scores.append(p_abn_raw(patient))
                labels.append(is_positive[patient])
        s = np.clip(np.array(scores), 1e-7, 1 - 1e-7); y = np.array(labels)
        loss = float(-(y*np.log(s) + (1-y)*np.log(1-s)).mean())
        auroc = roc_auc_score(labels, scores) if len(set(labels)) > 1 else float("nan")
        return auroc, loss

    best, best_loss, best_epoch, best_state, waited = -1.0, float("inf"), 0, None, 0
    for epoch in range(1, EPOCHS_MIL+1):
        mil.train(); losses = []
        order = [p for pair in zip(generator.sample(positives, len(positives)),
                                   generator.choices(negatives, k=len(positives))) for p in pair]
        for patient in order:
            features = cache[patient]
            index = torch.randperm(len(features), generator=torch.Generator().manual_seed(
                generator.randrange(10**9)))[:BAG]
            bag = features[index].to(DEVICE)
            output = mil(bag)
            target = torch.tensor(float(is_positive[patient]), device=DEVICE)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(output["logit"], target)
            loss = loss + 0.1 * attention_entropy_penalty(output["entropy"], len(bag))
            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            losses.append(loss.item())

        auroc, val_loss = evaluate_on(val_patients)
        run.metric("mil_head", epoch=epoch, loss=float(np.mean(losses)),
                   mil_val_auroc=auroc, mil_val_loss=val_loss)
        improved = auroc > best            # amendment 5a: AUROC, strict improvement
        if epoch % 5 == 0 or improved:
            print(f"epoch {epoch:3d}  loss {np.mean(losses):.4f}  mil_val AUROC {auroc:.4f}  "
                  f"val-loss {val_loss:.4f}"
                  + ("   <- best" if improved else f"   (patience {waited+1}/{PATIENCE_MIL})"))
        if improved:
            best, best_loss, best_epoch, waited = auroc, val_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in mil.state_dict().items()}
            run.checkpoint("mil_head", best_state, epoch=epoch, keep_last=1)
        else:
            waited += 1
            if waited >= PATIENCE_MIL:
                print(f"early stop at epoch {epoch}; best was epoch {best_epoch} "
                      f"(AUROC {best:.4f}, val-loss {best_loss:.4f})")
                break

    mil.load_state_dict(best_state); mil.to(DEVICE)
    torch.save(best_state, MIL_CKPT)
    run.result("mil_head_training.json",
               {"best_epoch": best_epoch, "best_mil_val_auroc": best, "epochs_run": epoch,
                "max_epochs": EPOCHS_MIL, "patience": PATIENCE_MIL,
                "monitor": "AUROC on mil_val patients (never dev_fit)",
                "n_val_patients": len(val_patients)})
    print(f"saved {MIL_CKPT} (best epoch {best_epoch}, mil_val AUROC {best:.4f})")

## 7. Resolving [FIT] and [REF] — the only place thresholds are set

* **[FIT]** on `dev_fit` patients, disjoint from `train_mil`.
* **[REF]** as `max(clinical floor, 99th percentile of the reference population)`, computed
  on **control patients of `dev_cal` only**.

cAItomorph is not touched. The resolved values are written back to `decision_grid.yaml` as
a dated amendment and re-hashed into `PREREGISTRATION.sha256`.

In [ ]:
def _operating_point(fpr, cuts, max_fpr=0.05):
    """Lowest threshold whose FPR stays <= max_fpr: maximum sensitivity at spec >= 0.95.

    NOT cuts[np.argmax(fpr <= max_fpr)]: roc_curve's first threshold is +inf with FPR 0, so
    argmax always returns index 0 and the operating point becomes +inf - sensitivity 0,
    every session "non-leukemic". Caught before the first real run of this section.
    """
    index = np.where(fpr <= max_fpr)[0][-1]
    value = float(cuts[index])
    assert np.isfinite(value), "operating point is not finite - check the ROC inputs"
    return value

import yaml
from sklearn.metrics import roc_curve

thresholds = load_thresholds(REPO/"src/aster_block2/decision_grid.yaml")
resolved = {}

# --- tau_abn: specificity >= 0.95 on dev_fit ---------------------------------
mil.eval()
scores = [p_abn_raw(p) for p in fit_patients]
labels = [is_positive[p] for p in fit_patients]
fpr, tpr, cuts = roc_curve(labels, scores)
# RAW scale here. It is mapped to the calibrated scale exactly once, in the temperature cell,
# from this stored raw value - so re-running either cell can never apply the mapping twice.
resolved["p_abn_raw"] = float(_operating_point(fpr, cuts))
resolved["p_abn"] = resolved["p_abn_raw"]
print(f"tau_abn (raw) = {resolved['p_abn_raw']:.4f}  (specificity >= 0.95 on {len(fit_patients)} dev_fit patients)")

# --- theta_APL: specificity >= 0.95 among AML on dev_fit ---------------------
apl_fraction, apl_label = [], []
for patient in fit_patients:
    if patient_label[patient] == "control": continue
    counts = predict_counts(mll[mll.patient_id==patient].corpus_path.tolist())
    n_c = sum(counts[c] for c in CELL_CLASSES if c != "other_artifact") or 1
    apl_fraction.append(counts["promyelocyte_abnormal"]/n_c)
    apl_label.append(int(patient_label[patient] == "PML_RARA"))
if sum(apl_label) >= 3:
    fpr, tpr, cuts = roc_curve(apl_label, apl_fraction)
    try:
        resolved["apl_abn_promy"] = float(_operating_point(fpr, cuts))
    except AssertionError:
        # abn_promy is predicted almost never (recall ~3 %), so the APL ROC can be degenerate.
        # Harmless: the quantifier declares abn_promy_frac unquantifiable and R1a returns
        # indeterminate before this threshold is ever consulted. Keep the clinical floor.
        resolved["apl_abn_promy"] = 0.10
        run.alert("theta_APL: degenerate ROC (abnormal promyelocytes almost never predicted); "
                  "kept the 0.10 floor. R1a is unquantifiable anyway.")
else:
    resolved["apl_abn_promy"] = 0.10   # too few APL patients in dev_fit: keep the floor
print(f"theta_APL = {resolved['apl_abn_promy']:.4f}  ({sum(apl_label)} PML_RARA / {len(apl_label)} AML in dev_fit)")

# --- [REF] reference intervals on CONTROL patients only ----------------------
# Amendment 1b: the population is the AML-MLL controls of dev_fit U dev_cal (~24
# patients), disjoint from train_mil. PBC has no patient identifiers and the reference
# intervals are per-patient proportions, so PBC trains the cell head but not this.
reference_patients = [p for p in (fit_patients + cal_patients) if patient_label[p] == "control"]
# The reference intervals are computed on the SAME scale the grid tests them on: CORRECTED
# proportions (quantifier), over N_c = the 15 leukocyte classes (other_artifact excluded).
# Raw argmax fractions would put each threshold on a different scale from the value it is
# compared with - correction inflates by ~1/(TPR-FPR), so a control sitting exactly at the
# raw p99 would exceed its own threshold once corrected.
from aster_block2.grid import QUANTITIES as _Q, CELL_CLASSES as _LEUKOCYTES
control_fractions = {"ig_frac": [], "baso_frac": [], "lymph_frac": [], "smudge_frac": [], "atypical_frac": []}
for patient in reference_patients:
    counts = predict_counts(mll[mll.patient_id==patient].corpus_path.tolist())
    n_c = sum(counts[c] for c in _LEUKOCYTES) or 1
    for quantity in control_fractions:
        k = sum(counts[c] for c in _Q[quantity])
        corrected, _variance, _ok = quantifier.correct(quantity, k, n_c)
        control_fractions[quantity].append(corrected)

percentile = thresholds.reference_percentile
floors = {"ig_cml": ("ig_frac", 0.10), "baso_cml": ("baso_frac", 0.02),
          "lymph_cll": ("lymph_frac", 0.50), "smudge_cll": ("smudge_frac", 0.02),
          "atypical_reactive": ("atypical_frac", 0.10)}
n_controls = len(reference_patients)
reference_report = {}
print(f"{'threshold':<20}{'floor':>8}{'p99':>10}{'adopted':>10}   term")
for name, (quantity, floor) in floors.items():
    p99 = float(np.percentile(control_fractions[quantity], percentile)) if n_controls else 0.0
    resolved[name] = max(floor, p99)
    term = "floor" if floor >= p99 else "empirical"
    reference_report[name] = {"clinical_floor": floor, "empirical_percentile": p99,
                              "n_reference_patients": n_controls, "adopted": resolved[name],
                              "adopted_term": term}
    print(f"{name:<20}{floor:>8.3f}{p99:>10.4f}{resolved[name]:>10.4f}   {term}")
json.dump(reference_report, open(WORK/"results/reference_intervals.json","w"), indent=2)
run.result("reference_intervals.json", reference_report)
print(f"\nreference population: {n_controls} control patients (dev_fit U dev_cal)")
print("Amendment 1b declared that the floor is expected to dominate at this n - report both terms either way.")


In [ ]:
# --- blast lineage band, on PSEUDO-BAGS (PREREGISTRATION.md §5.2, amendment 8) ---------
# lineage_post is a SESSION aggregate at inference - sum of lymphoblast probability over
# the sum of both, taken over the cells predicted as blasts - so the band is fitted on bags
# built the same way, not on single cells. Fitted per cell, the band came out as
# [0.01, 0.31]: a myeloid call would have needed almost zero lymphoid mass across hundreds
# of blasts, and nearly every AML session would have landed in `lineage_indeterminate`.
lymphoblast_index, myeloblast_index = CLASS_INDEX["lymphoblast"], CLASS_INDEX["myeloblast"]
blast_rows = cell_rows[(cell_rows.split=="cell_val") &
                       cell_rows.cell_labels.isin(["lymphoblast","myeloblast"])]
loader = DataLoader(CellDataset(blast_rows, DATA/"corpus", eval_tf), batch_size=128, num_workers=2)
p_lymph, p_myelo, pred_blast, is_lymph = [], [], [], []
with torch.inference_mode():
    for images, allowed, _ in loader:
        probs = cell_head(encoder(images.to(DEVICE)))["probabilities"].cpu()
        pred = probs.argmax(-1)
        p_lymph += probs[:, lymphoblast_index].tolist()
        p_myelo += probs[:, myeloblast_index].tolist()
        pred_blast += ((pred == lymphoblast_index) | (pred == myeloblast_index)).tolist()
        is_lymph += allowed[:, lymphoblast_index].bool().tolist()
p_lymph, p_myelo = np.array(p_lymph), np.array(p_myelo)
pred_blast, is_lymph = np.array(pred_blast), np.array(is_lymph)

# Bag size is DERIVED, not chosen: a session reaches the lineage decision only at the
# pattern tier (N_c >= 200 [STD]) and only once blasts reach 20 % [WHO], so the smallest
# blast population that ever gets a lineage call is 200 x 0.20 = 40. The smallest bag is
# also the noisiest, which makes the band conservative for every larger session.
LINEAGE_BAG = int(thresholds.tier_pattern * thresholds.blast_acute)     # = 40
LINEAGE_BAGS, LINEAGE_ERROR = 1000, 0.05

def _bag_posteriors(pool, n_bags=LINEAGE_BAGS, bag=LINEAGE_BAG):
    """Session-level lineage_post of pseudo-bags drawn from one lineage's blasts.

    LINEAGE_BAG blasts = 200 [STD] x 20 % [WHO]: the smallest blast population that can
    reach R1b. Smaller bags are noisier, so the band is conservative for real sessions.
    """
    rng = np.random.default_rng(SEED)
    pool = pool[pred_blast[pool]]             # as at inference: only predicted blasts enter
    out = []
    for _ in range(n_bags):
        idx = rng.choice(pool, size=bag, replace=True)
        l, m = p_lymph[idx].sum(), p_myelo[idx].sum()
        out.append(l / (l + m + 1e-9))
    return np.array(out)

lymph_bags = _bag_posteriors(np.where(is_lymph)[0])
myelo_bags = _bag_posteriors(np.where(~is_lymph)[0])
# Each bound caps one error at 5 %, the same convention as tau_abn at specificity >= 0.95.
theta_lo = float(np.percentile(lymph_bags, 100 * LINEAGE_ERROR))        # <= 5 % of ALL -> myeloid
theta_hi = float(np.percentile(myelo_bags, 100 * (1 - LINEAGE_ERROR)))  # <= 5 % of AML -> lymphoid
separable = theta_lo >= theta_hi
if separable:                                  # no overlap: a single cut, no band
    theta_lo = theta_hi = (theta_lo + theta_hi) / 2
resolved["lineage_lo"], resolved["lineage_hi"] = theta_lo, theta_hi

print(f"myeloid bags  lineage_post: median {np.median(myelo_bags):.3f}  p95 {np.percentile(myelo_bags,95):.3f}")
print(f"lymphoid bags lineage_post: median {np.median(lymph_bags):.3f}  p5  {np.percentile(lymph_bags,5):.3f}")
print(f"-> {'SEPARABLE, single cut' if separable else 'overlap, indeterminate band'} "
      f"[{theta_lo:.3f}, {theta_hi:.3f}]   (bags of {LINEAGE_BAG} predicted blasts, "
      f"{is_lymph.sum()} lymphoblast / {(~is_lymph).sum()} myeloblast cells)")
lineage_report = {}
for name, bags in (("AML_bags", myelo_bags), ("ALL_bags", lymph_bags)):
    shares = {"myeloid": float(np.mean(bags <= theta_lo)), "lymphoid": float(np.mean(bags >= theta_hi)),
              "indeterminate": float(np.mean((bags > theta_lo) & (bags < theta_hi)))}
    lineage_report[name] = shares
    print(f"   {name}: myeloid {shares['myeloid']:.1%}  lymphoid {shares['lymphoid']:.1%}  "
          f"indeterminate {shares['indeterminate']:.1%}")
run.result("lineage_band.json", {"theta_lo": theta_lo, "theta_hi": theta_hi, "separable": separable,
                                 "bag_size": LINEAGE_BAG, "n_bags": LINEAGE_BAGS,
                                 "error_per_side": LINEAGE_ERROR, "shares": lineage_report})

In [ ]:
# --- OOD domain gate, N_c/N floor, and MIL temperature -----------------------
from aster_block2.ood import DomainGate
from aster_block2.calibration import TemperatureScaler, expected_calibration_error

# OOD: fitted on in-domain dev bags at a 99% in-domain pass rate  [FIT]
in_domain = torch.cat([cache[p] for p in fit_patients]).numpy()
cal_bags  = [cache[p].numpy() for p in cal_patients]
domain_gate = DomainGate.fit(in_domain, cal_bags, pass_rate=0.99)
resolved["ood_mahalanobis"] = domain_gate.threshold
domain_gate.save(WORK/"checkpoints/ood_stats.npz")
print(f"OOD threshold = {domain_gate.threshold:.1f}  (99th percentile of {len(cal_bags)} in-domain dev_cal bags)")

# N_c/N floor: 1st percentile of the dev distribution  [FIT]
classified_fractions = []
for patient in fit_patients + cal_patients:
    counts = predict_counts(mll[mll.patient_id==patient].corpus_path.tolist())
    n_all = sum(counts.values())
    n_leu = sum(counts[c] for c in CELL_CLASSES if c != "other_artifact")
    if n_all: classified_fractions.append(n_leu / n_all)
resolved["min_classified_frac"] = float(np.percentile(classified_fractions, 1))
print(f"N_c/N floor = {resolved['min_classified_frac']:.3f}  "
      f"(1st percentile of {len(classified_fractions)} dev sessions)")

# MIL temperature: fitted on dev_cal, never on train_mil, never on cAItomorph  [FIT]
mil.eval()
cal_scores = [p_abn_raw(p) for p in cal_patients]
cal_labels = [is_positive[p] for p in cal_patients]
calibrator = TemperatureScaler().fit(cal_scores, cal_labels)
# tau_abn was fitted on RAW MIL probabilities, but at inference it is compared with the
# CALIBRATED P_abn. Temperature scaling is monotone, so the operating point maps across
# exactly - it just has to be expressed on the same scale as what it is compared to.
resolved["p_abn"] = calibrator.apply(resolved["p_abn_raw"])   # from RAW, every time: idempotent
print(f"tau_abn on the calibrated scale: {resolved['p_abn']:.4f}")
ece_before = expected_calibration_error(cal_scores, cal_labels)
ece_after  = expected_calibration_error([calibrator.apply(s) for s in cal_scores], cal_labels)
resolved["mil_temperature"] = calibrator.temperature
print(f"temperature = {calibrator.temperature:.3f}   ECE {ece_before:.4f} -> {ece_after:.4f}  "
      f"(n={len(cal_patients)} dev_cal patients)")
print("Temperature scaling cannot change the ranking, so AUROC is untouched by construction.")

In [ ]:
# --- write the amendment ------------------------------------------------------
STAMP = time.strftime("%Y-%m-%d")
grid_path = REPO/"src/aster_block2/decision_grid.yaml"
grid = yaml.safe_load(grid_path.read_text())
grid.setdefault("amendments", []).append({
    "date": STAMP,
    "reason": "Phase 1 resolution of the [REF] and [FIT] placeholders",
    "fitted_on": {"FIT": "aml_mll dev_fit patients", "REF": f"{n_controls} aml_mll dev_cal control patients"},
    "never_used": "caitomorph",
    "values": {k: round(v, 6) for k, v in resolved.items()},
})
grid_path.write_text(yaml.safe_dump(grid, sort_keys=False, allow_unicode=True))
json.dump(resolved, open(WORK/"results/resolved_thresholds.json","w"), indent=2)
run.result("resolved_thresholds.json", resolved)

for name, value in resolved.items(): setattr(thresholds, name, value)
missing = thresholds.unresolved()
print("unresolved:", missing if missing else "none - the grid is executable")
print(json.dumps(resolved, indent=2))

In [ ]:
# --- pre-flight: is each criterion assertable at all, given the measured error rates? ---
# A threshold sitting near the classifier's own false-positive rate cannot be asserted: the
# correction attributes the observation to error. Better to know it here than to discover
# it as a wall of `indeterminate` in section 9.
from aster_block2.proportion_test import test_proportion

# The thresholds the grid ACTUALLY applies - the resolved [REF]/[FIT] values - never the
# clinical floors. The first version tested the floors and reported that R3 needs 3.5 %
# observed smudge cells, when the resolved threshold (13.7 %) needs about 15 %.
CRITERIA = [("blast_frac", thresholds.blast_acute), ("ig_frac", thresholds.ig_cml),
            ("baso_frac", thresholds.baso_cml), ("lymph_frac", thresholds.lymph_cll),
            ("smudge_frac", thresholds.smudge_cll), ("mono_frac", thresholds.mono_cmml),
            ("abn_promy_frac", thresholds.apl_abn_promy),
            ("atypical_frac", thresholds.atypical_reactive)]

print(f"{'criterion':<16}{'theta':>7}{'FPR':>7}   observed fraction needed to assert")
for name, theta in CRITERIA:
    rate = quantifier.rates.get(name)
    fpr = rate.fpr if rate else float("nan")
    needed = None
    for n_c in (200, 500):
        for k in range(0, n_c + 1):
            eff_k, eff_n, ok = quantifier.effective_counts(name, k, n_c)
            if ok and test_proportion(eff_k, eff_n, theta, thresholds.gamma).value == "assert":
                needed = (n_c, k / n_c); break
        if needed and needed[0] == n_c:
            print(f"{name:<16}{theta:>7.2f}{fpr:>7.3f}   N_c={n_c}: {needed[1]:>6.1%}"
                  + ("   <- above 50 %, effectively unassertable" if needed[1] > 0.5 else ""))
            needed = None
        else:
            print(f"{name:<16}{theta:>7.2f}{fpr:>7.3f}   N_c={n_c}: UNREACHABLE"
                  f"   <- this criterion can never fire; report it, do not retune")

## 8. Session scoring

One function: bag of crops → cell counts, `P_abn`, lineage posterior, OOD score → the
frozen grid. This is exactly what `integration/` will run on the Jetson.

In [ ]:
# The scorer lives in src/aster_block2/inference.py so that this notebook and the Jetson
# run the SAME code. No second implementation to drift - the same discipline as the
# crop->tensor contract, one level up.
from aster_block2.inference import SessionScorer

scorer = SessionScorer(encoder=encoder, cell_head=cell_head, mil_head=mil,
                       thresholds=thresholds, domain_gate=domain_gate,
                       calibrator=calibrator, quantifier=quantifier,
                       device=DEVICE, bag_size=BAG, seed=SEED)

def score_session(paths, session_id="session", n_fields=0):
    """Thin wrapper keeping the (result, detail) shape used by sections 9-11."""
    result, detail = scorer.score(list(paths), session_id=session_id, number_of_fields=n_fields)
    detail["p_abn"] = result.uncertainty["p_abn"]
    detail["ood"] = result.uncertainty["ood_score"]
    result.label_obj = result          # sections below read .label / .tier / .reasons
    return result, detail

print("session scorer ready (shared with integration/)")

## 9. Primary test — cAItomorph, 409 patients

**The first time cAItomorph is read.** One row per `diagnosis_fine`, abstention rate per
row, and the tier the patient reached. Nothing here changes a threshold.

In [ ]:
import pickle
cai = manifest[manifest.source == "caitomorph"].copy()
cai["diagnosis_fine"] = cai.bag_label
records, cai_inputs = [], {}
for patient, group in cai.groupby("patient_id"):
    paths = [DATA/"corpus"/p for p in group.corpus_path.tolist()]
    result, detail = score_session(paths, patient)
    # The grid's INPUTS per patient, so section 11 re-runs only the grid - never the encoder.
    # The sensitivity arms then see strictly identical inputs, and cost seconds, not hours.
    cai_inputs[patient] = {"counts": detail["counts"], "n_localized": result.number_of_detected_wbc,
                           "p_abn": detail["p_abn"], "lineage_post": detail["lineage_post"],
                           "ood": detail["ood"], "domain_verdict": detail["domain_verdict"]}
    records.append({"patient_id": patient, "diagnosis_fine": group.diagnosis_fine.iloc[0],
                    "diagnosis_coarse": json.loads(group.extra.iloc[0]).get("diagnosis_coarse"),
                    "label": result.label, "tier": result.tier, "flags": "|".join(result.flags),
                    # SessionResult names it number_of_classified_leukocytes; `n_classified`
                    # is the GridResult name and does not exist here.
                    "n_classified": result.number_of_classified_leukocytes,
                    "p_abn": detail["p_abn"], "ood": detail["ood"],
                    "reasons": " ; ".join(result.reasons)})
test_frame = pd.DataFrame(records)
test_frame.to_csv(WORK/"results/caitomorph_409_predictions.csv", index=False)
run.result("caitomorph_409_predictions.csv", test_frame)
pickle.dump(cai_inputs, open(DATA/"cai_inputs.pkl", "wb"))
print(len(test_frame), "patients scored")
print("out_of_domain:", (test_frame.label == "out_of_domain").sum(),
      "| indeterminate:", (test_frame.label == "indeterminate").sum())

In [ ]:
crosstab = pd.crosstab(test_frame.diagnosis_fine, test_frame.label)
crosstab["n"] = crosstab.sum(1)
crosstab["abstention"] = (test_frame.groupby("diagnosis_fine")
                          .label.apply(lambda s: (s=="indeterminate").mean()).round(3))
print(crosstab.to_string())
crosstab.to_csv(WORK/"results/caitomorph_409_confusion.csv")
run.result("caitomorph_409_confusion.csv", crosstab.reset_index())
print("\ntiers reached:", test_frame.tier.value_counts().to_dict())

# Tier A endpoint: specificity reported separately against donors and against reactive
from sklearn.metrics import confusion_matrix as cm
acute = test_frame.label.str.startswith("acute_blastic")
for negative_group in ("Stem cell donor", "Reactive changes"):
    subset = test_frame[test_frame.diagnosis_fine == negative_group]
    false_positive = acute[subset.index].sum()
    print(f"specificity vs {negative_group:<18} "
          f"{1-false_positive/len(subset):.3f}  ({false_positive} acute calls / {len(subset)})")
for positive_group in ("AML", "ALL", "AL"):
    subset = test_frame[test_frame.diagnosis_fine == positive_group]
    if not len(subset): continue
    print(f"sensitivity (acute call) on {positive_group:<6} "
          f"{acute[subset.index].mean():.3f}  (n={len(subset)})")

In [ ]:
# --- section 3.4: secondary analysis restricted to the reference tier ---------
tier_r = test_frame[test_frame.n_classified >= thresholds.tier_reference]
print(f"tier R (N_c >= {thresholds.tier_reference}): {len(tier_r)}/{len(test_frame)} patients")
if len(tier_r):
    print(pd.crosstab(tier_r.diagnosis_fine, tier_r.label).to_string())
    pd.crosstab(tier_r.diagnosis_fine, tier_r.label).to_csv(WORK/"results/caitomorph_tierR_confusion.csv")

# --- falsifier 6.5: tier-P abstention above 50% -------------------------------
pattern_tier = test_frame[test_frame.tier.isin(["pattern", "reference"])]
abstention = (pattern_tier.label == "indeterminate").mean() if len(pattern_tier) else 1.0
print(f"\ntier-P abstention rate: {abstention:.1%}")
if abstention > 0.50:
    run.alert(f"criterion 6.5 MET: tier-P abstention {abstention:.1%} > 50 %. The grid is "
              f"under-powered at the available session sizes; the tier-S screening "
              f"statement becomes the headline claim. Do not retune.", severity="FALSIFIER")

In [ ]:
# Tier C, exploratory, wide CI, excluded from the abstract
for label, groups in (("chronic_myeloid_pattern", ["CML","MPN","CMML","MDS / MPN","ET","PV"]),
                      ("chronic_lymphoid_pattern", ["B-cell neoplasm","HCL","T-cell neoplasm","MM"])):
    print(f"\n{label}")
    for group in groups:
        subset = test_frame[test_frame.diagnosis_fine == group]
        if not len(subset): continue
        print(f"  {group:<18} {(subset.label==label).sum():>3}/{len(subset):<3} fired")
    donors = test_frame[test_frame.diagnosis_fine == "Stem cell donor"]
    fired_on_donors = (donors.label == label).sum()
    print(f"  {'Stem cell donor':<18} {fired_on_donors:>3}/{len(donors):<3} fired  <- falsification check 6.5")

## 9b. Amendment 9 — session-level blast recalibration (post hoc, secondary)

Declared and hashed in PREREGISTRATION.md before it was run. Fitted on the 189 AML-MLL patients
against their manual differential; cAItomorph is replayed from `cai_inputs` exactly once.
The primary result above is not replaced: both are reported side by side.


In [ ]:
# --- amendment 9 (1/2): session-level blast recalibration on the 189 AML-MLL patients ---
# PREREGISTRATION.md amendment 9. cAItomorph takes no part in this cell.
import numpy as np
from sklearn.model_selection import StratifiedKFold
from statsmodels.stats.proportion import proportion_confint
from aster_block2.session_calibration import SessionCalibration, RecalibratedQuantifier
from aster_block2.grid import QUANTITIES as _Q, CELL_CLASSES as _LEUKOCYTES
from aster_block2.proportion_test import test_proportion

if "cache" not in globals():
    cache = torch.load(DATA/"features_mll.pt")          # written by section 6
mll["differential"] = mll.extra.map(lambda s: json.loads(s).get("differential", {}))
cell_head.eval()
BLAST = set(_Q["blast_frac"])
rows = []
for patient, group in mll.groupby("patient_id"):
    manual, label = group.differential.iloc[0], group.bag_label.iloc[0]
    total = float(manual.get("pb_total") or 0)
    if not total or patient not in cache: continue
    with torch.inference_mode():   # same argmax over all crops as SessionScorer
        names = [CELL_CLASSES[i] for i in cell_head(cache[patient].to(DEVICE))["logits"].argmax(-1).cpu().tolist()]
    n_c = sum(name in _LEUKOCYTES for name in names)
    k = sum(name in BLAST for name in names)
    # WHO blast equivalents: in APL the counted promyelocytes are the abnormal ones
    blasts = float(manual.get("pb_myeloblast") or 0)
    if label == "PML_RARA": blasts += float(manual.get("pb_promyelocyte") or 0)
    rows.append({"patient_id": patient, "label": label,
                 "stratum": "control" if label == "control" else "aml",
                 "k": k, "n_c": n_c, "q": k / n_c, "manual": blasts / total, "manual_total": total})
cal_frame = pd.DataFrame(rows)

blast_cal = SessionCalibration.fit(cal_frame.k, cal_frame.n_c, cal_frame.manual, cal_frame.stratum,
                                   manual_total=cal_frame.manual_total, seed=SEED)
d = blast_cal.diagnostics
fmt = lambda pair: f"[{pair[0]:.4f}, {pair[1]:.4f}]"
print(f"patients                 {d['n_per_stratum']}")
print(f"observed blast, median   " + "  ".join(f"{s} {v:.3f}" for s, v in d['observed_median_per_stratum'].items()))
print(f"a  session FPR           {blast_cal.intercept:.4f}  IC95 {fmt(d['intercept_ci95'])}")
print(f"b  session TPR - FPR     {blast_cal.slope:.4f}  IC95 {fmt(d['slope_ci95'])}")
print(f"phi dispersion           {blast_cal.dispersion:.2f}   per stratum " + "  ".join(f"{s} {v:.2f}" for s, v in d['dispersion_per_stratum'].items()))
print(f"regression dilution      {d['reliability_ratio']:.3f}   (reported, not corrected)")
print(f"9c-1 applicable          {blast_cal.usable}   (lower IC95 of b >= 0.05)")

recal = RecalibratedQuantifier(quantifier, {"blast_frac": blast_cal})
recal.save(WORK/"checkpoints/quantifier_amendment9.json")
run.result("amendment9_blast_calibration.json", {"calibration": vars(blast_cal), "usable": blast_cal.usable})
run.result("amendment9_mll_patients.csv", cal_frame)

# --- 9c-3: 5-fold patient-level cross-validation on AML-MLL -----------------
def blast_verdicts(q_obj, k, n):
    ke, ne, ok = q_obj.effective_counts("blast_frac", int(k), int(n))
    if not ok: return "indeterminate", "indeterminate"
    return (test_proportion(ke, ne, thresholds.blast_normal, thresholds.gamma).value,
            test_proportion(ke, ne, thresholds.blast_acute, thresholds.gamma).value)
cv_rows = []
for train_idx, test_idx in StratifiedKFold(5, shuffle=True, random_state=SEED).split(cal_frame, cal_frame.label):
    tr = cal_frame.iloc[train_idx]
    fold_q = RecalibratedQuantifier(quantifier, {"blast_frac": SessionCalibration.fit(
        tr.k, tr.n_c, tr.manual, tr.stratum, seed=SEED)})
    for _, r in cal_frame.iloc[test_idx].iterrows():
        (b5, b20), (a5, a20) = blast_verdicts(quantifier, r.k, r.n_c), blast_verdicts(fold_q, r.k, r.n_c)
        cv_rows.append({"patient_id": r.patient_id, "stratum": r.stratum, "manual": r.manual,
                        "ge5_before": b5, "ge20_before": b20, "ge5_after": a5, "ge20_after": a20})
cv = pd.DataFrame(cv_rows)
run.result("amendment9_mll_cv.csv", cv)
def share(series, value):
    k, n = int((series == value).sum()), len(series)
    lo, hi = proportion_confint(k, n, method="wilson")
    return f"{k}/{n} = {k/n:.2f} [{lo:.2f}, {hi:.2f}]"
ctrl, acute = cv[cv.stratum == "control"], cv[(cv.stratum == "aml") & (cv.manual >= 0.20)]
print(f"\n9c-3  5-fold CV, AML-MLL                  {'cell-level quantifier':<30}  amendment 9")
print(f"controls: blast >= 5 %  REJECTED           {share(ctrl.ge5_before, 'reject'):<30}  {share(ctrl.ge5_after, 'reject')}")
print(f"controls: blast >= 20 % ASSERTED (harm)    {share(ctrl.ge20_before, 'assert'):<30}  {share(ctrl.ge20_after, 'assert')}")
print(f"AML manual >= 20 %: blast >= 20 % ASSERTED {share(acute.ge20_before, 'assert'):<30}  {share(acute.ge20_after, 'assert')}")


In [ ]:
# --- amendment 9 (2/2): replay check, then cAItomorph ONCE -------------------------
# Run only after reading the output of (1/2). Nothing below is refitted, whatever it gives.
import pickle
from aster_block2.grid import evaluate
from aster_block2.ood import NOT_VALIDATED, UNKNOWN
if not blast_cal.usable:
    raise SystemExit("9c-1: amendment 9 NOT APPLICABLE (IC95 of b reaches 0.05). Report it; stop here.")
if "cai_inputs" not in globals():
    cai_inputs = pickle.load(open(DATA/"cai_inputs.pkl", "rb"))

def replay(q_obj):   # the SessionScorer grid call, from the stored per-patient inputs
    out = {}
    for patient, x in cai_inputs.items():
        verdict = x["domain_verdict"]
        r = evaluate(x["counts"], thresholds, n_localized=x["n_localized"], p_abn=x["p_abn"],
                     lineage_post=x["lineage_post"],
                     ood_score=x["ood"] if verdict in (None, UNKNOWN) else None, quantifier=q_obj)
        if verdict == NOT_VALIDATED: r.label = "out_of_domain"
        out[patient] = r
    return out

primary = test_frame.set_index("patient_id").label
frozen = replay(quantifier)
mismatch = [p for p, r in frozen.items() if r.label != primary[p]]
assert not mismatch, f"9c-2 FAILED: replay differs from the primary run on {len(mismatch)} patients {mismatch[:5]} - stop"
print(f"9c-2 replay: {len(frozen)}/{len(primary)} primary labels reproduced exactly\n")

amended = replay(recal)
cmp = test_frame[["patient_id", "diagnosis_fine", "label"]].rename(columns={"label": "frozen"})
cmp["amended"] = cmp.patient_id.map(lambda p: amended[p].label)
cmp["amended_reasons"] = cmp.patient_id.map(lambda p: " ; ".join(amended[p].reasons))
run.result("amendment9_caitomorph_409.csv", cmp)

leukaemic = lambda s: s.str.startswith("acute_blastic") | s.str.startswith("chronic_")
def wilson(mask):
    k, n = int(mask.sum()), len(mask)
    lo, hi = proportion_confint(k, n, method="wilson")
    return f"{k}/{n} = {k/n:.3f} [{lo:.3f}, {hi:.3f}]"
don = cmp[cmp.diagnosis_fine == "Stem cell donor"]; rea = cmp[cmp.diagnosis_fine == "Reactive changes"]
acu = cmp[cmp.diagnosis_fine.isin(["AML", "ALL", "AL"])]
metrics = [("non_leukemic, donors", lambda c: don[c] == "non_leukemic"),
           ("non_leukemic, reactive", lambda c: rea[c] == "non_leukemic"),
           ("specificity vs donors (no leukaemic label)", lambda c: ~leukaemic(don[c])),
           ("specificity vs reactive", lambda c: ~leukaemic(rea[c])),
           ("sensitivity acute (AML/ALL/AL -> acute_blastic)", lambda c: acu[c].str.startswith("acute_blastic")),
           ("indeterminate, all patients", lambda c: cmp[c] == "indeterminate")]
report = {}
print(f"{'':<48}{'frozen (primary)':<32}amendment 9 (post hoc)")
for name, mask in metrics:
    report[name] = {"frozen": wilson(mask("frozen")), "amended": wilson(mask("amended"))}
    print(f"{name:<48}{report[name]['frozen']:<32}{report[name]['amended']}")
run.result("amendment9_metrics.json", report)

changed = cmp[cmp.frozen != cmp.amended]
print(f"\n{len(changed)} labels changed:")
print(changed.groupby(["frozen", "amended"]).size().to_string())
harm = cmp[cmp.diagnosis_fine.isin(["Stem cell donor", "Reactive changes"])
           & leukaemic(cmp.amended) & ~leukaemic(cmp.frozen)]
print(f"\ndonors / reactive newly labelled leukaemic: {len(harm)}")
if len(harm):
    print(harm[["patient_id", "diagnosis_fine", "frozen", "amended"]].to_string(index=False))
    run.alert(f"amendment 9: {len(harm)} donor/reactive patients newly labelled leukaemic",
              detail=harm.to_string(index=False))
print("\n", pd.crosstab(cmp.diagnosis_fine, cmp.amended).to_string())


## 10. ×40 stress test

The figure that justifies the redesign. Old block 2 on this material: `AML`, 5/5 folds,
calibrated p = 0.986. Declared expectation: `out_of_domain` or `indeterminate`.

In [ ]:
x40_paths = sorted((DATA/"x40_sessions").rglob("crops/*.png"))
print(len(x40_paths), "x40 crops")

stress = []
CHUNK = 200   # split into sessions of 200 crops = the diagnostic differential tier
for start in range(0, len(x40_paths), CHUNK):
    chunk = x40_paths[start:start+CHUNK]
    if len(chunk) < thresholds.tier_screening: continue
    result, detail = score_session(chunk, f"x40_{start//CHUNK:03d}")
    stress.append({"session": f"x40_{start//CHUNK:03d}", "n": len(chunk), "label": result.label,
                   "tier": result.tier, "ood": detail["ood"], "p_abn": detail["p_abn"],
                   "reasons": " ; ".join(result.reasons)})
stress_frame = pd.DataFrame(stress)
stress_frame.to_csv(WORK/"results/x40_stress.csv", index=False)
run.result("x40_stress.csv", stress_frame)
print(stress_frame.to_string())

withheld = stress_frame.label.isin(["out_of_domain","indeterminate","insufficient_evidence"]).mean()
print(f"\nwithheld: {withheld:.1%} of x40 sessions")
print("old block 2 on the same material: AML, 5/5 folds, p=0.986")
if withheld < 1.0:
    run.alert(f"criterion 6.5 MET: {(1-withheld):.1%} of x40 sessions received a confident "
              f"class. The OOD gate failed; block 2 stays scoped out of the deployment "
              f"domain.", severity="FALSIFIER")

## 11. Sensitivity analysis on the three [OPS] values

Declared in advance, reported whatever it shows. A grid that only works at its frozen
values is a fitted grid.

In [ ]:
# --- 11 (fast): sensitivity on the three [OPS] values, replayed from cai_inputs ------
# Every arm sees strictly identical grid inputs (no re-encoding): arms differ ONLY by
# the OPS value. The frozen arm must reproduce the 409 primary labels first.
import pickle
from statsmodels.stats.proportion import proportion_confint
from aster_block2.grid import evaluate
from aster_block2.ood import NOT_VALIDATED, UNKNOWN
if "cai_inputs" not in globals():
    cai_inputs = pickle.load(open(DATA/"cai_inputs.pkl", "rb"))
frame0 = test_frame.set_index("patient_id")

def _replay():
    out = {}
    for patient, x in cai_inputs.items():
        verdict = x["domain_verdict"]
        r = evaluate(x["counts"], thresholds, n_localized=x["n_localized"], p_abn=x["p_abn"],
                     lineage_post=x["lineage_post"],
                     ood_score=x["ood"] if verdict in (None, UNKNOWN) else None, quantifier=quantifier)
        if verdict == NOT_VALIDATED: r.label = "out_of_domain"
        out[patient] = r
    return out

def _summary(results, parameter, value):
    s = pd.DataFrame({"dx": frame0.diagnosis_fine,
                      "label": pd.Series({p: r.label for p, r in results.items()})})
    leuk = s.label.str.startswith("acute_blastic") | s.label.str.startswith("chronic_")
    don, rea = s.dx == "Stem cell donor", s.dx == "Reactive changes"
    acu = s.dx.isin(["AML", "ALL", "AL"])
    chronic = s.label.str.startswith("chronic_")
    return {"parameter": parameter, "value": value,
            "non_leuk_donors": f"{int((s.label[don] == 'non_leukemic').sum())}/{int(don.sum())}",
            "spec_donors": round(float((~leuk[don]).mean()), 3),
            "spec_reactive": round(float((~leuk[rea]).mean()), 3),
            "sens_acute": f"{int(s.label[acu].str.startswith('acute_blastic').sum())}/{int(acu.sum())}",
            "indeterminate": round(float((s.label == "indeterminate").mean()), 3),
            "chronic_calls": int(chronic.sum()),
            "chronic_on_donors": int((chronic & don).sum()),
            "chronic_on_CML_or_B": int((chronic & s.dx.isin(["CML", "B-cell neoplasm"])).sum()),
            "APL_flags": sum("APL_suspicion" in r.flags for r in results.values())}

base_gamma = thresholds.gamma
REF_NAMES = ("ig_cml", "baso_cml", "lymph_cll", "smudge_cll", "atypical_reactive")
base_ref = {name: getattr(thresholds, name) for name in REF_NAMES}

frozen_results = _replay()
mismatch = [p for p, r in frozen_results.items() if r.label != frame0.label[p]]
assert not mismatch, f"frozen arm differs from the primary run on {len(mismatch)} patients - stop"
print(f"frozen arm: {len(frozen_results)}/{len(frame0)} primary labels reproduced\n")

rows = []
try:
    for gamma in (0.80, 0.90, 0.95):
        thresholds.gamma = gamma
        rows.append(_summary(_replay(), "gamma", gamma))
    thresholds.gamma = base_gamma
    if "control_fractions" in globals():
        for percentile in (97.5, 99.0, 99.5):
            for name, (quantity, floor) in floors.items():
                setattr(thresholds, name, max(floor, float(np.percentile(control_fractions[quantity], percentile))))
            rows.append(_summary(_replay(), "reference_percentile", percentile))
    else:
        print("control_fractions not in memory (section 7 not run in this session): percentile arm skipped")
finally:
    thresholds.gamma = base_gamma
    for name, value in base_ref.items(): setattr(thresholds, name, value)

sensitivity_frame = pd.DataFrame(rows)
print(sensitivity_frame.to_string(index=False))
run.result("sensitivity_ops.csv", sensitivity_frame)

# R1a [OPS] - `abn_promy_frac > myeloblast_frac`: dropping it can only matter where the
# APL criterion itself ASSERTS. Count those sessions under the frozen values.
apl_asserts = sum(r.verdicts.get("apl") == "assert" for r in frozen_results.values())
print(f"\nR1a: APL criterion ASSERTED on {apl_asserts} sessions -> "
      + ("dropping the comparison changes nothing (abn_promy_frac unquantifiable)" if apl_asserts == 0
         else "see the dropped arm"))


## 12. Jetson kit (ONNX + golden set)

Exports what the Orin Nano needs: the encoder as ONNX (static batch 8, input `images`, the
deployed engines' convention), the heads, every resolved threshold at full precision, and a
golden set of 6 cAItomorph sessions with their outputs. The TensorRT engine is built **on the
Jetson** (`jetson/README_JETSON.md`), never here: engines are not portable.


In [ ]:
# --- 12. Jetson kit: everything the Orin Nano needs, produced from THIS run -------------
# Colab exports the models and a golden set; the engine itself is built ON the Jetson
# (jetson/README_JETSON.md). Nothing here changes a threshold or a result.
import hashlib, shutil
import onnxruntime as ort
from aster_block2.preprocess import build_eval_transform
from aster_block2.sampling import deterministic_bag

assert abs(thresholds.gamma - 0.90) < 1e-12, "thresholds drifted from the frozen state - restore them first"
KIT = run.root/"jetson_kit"; MODELS, GOLDEN = KIT/"models", KIT/"golden"
shutil.rmtree(KIT, ignore_errors=True)
MODELS.mkdir(parents=True); (GOLDEN/"sessions").mkdir(parents=True)
TRT_BATCH = 8   # static batch of the deployed engines (Software_Dev_Micro_Edge export_manifest.json)

# 1. encoder -> ONNX: static batch 8, input 'images', opset 17, classic exporter
encoder.eval().cpu()
dummy = torch.randn(TRT_BATCH, 3, 224, 224)
torch.onnx.export(encoder, dummy, str(MODELS/"encoder.onnx"), input_names=["images"],
                  output_names=["features"], opset_version=17, dynamo=False)
with torch.inference_mode():
    reference = encoder(dummy).numpy()
produced = ort.InferenceSession(str(MODELS/"encoder.onnx"), providers=["CPUExecutionProvider"]).run(
    None, {"images": dummy.numpy()})[0]
onnx_diff = float(np.abs(reference - produced).max())
encoder.to(DEVICE)
print(f"encoder.onnx  {(MODELS/'encoder.onnx').stat().st_size/1e6:.1f} MB   max |torch - onnx| = {onnx_diff:.2e}")
assert onnx_diff < 1e-3, "the ONNX graph does not reproduce the encoder"

# 2. weights (fp32 encoder kept: reference path + fallback), thresholds at full precision
cpu = lambda module: {k: v.detach().cpu() for k, v in module.state_dict().items()}
torch.save({"encoder": cpu(encoder), "cell_head": cpu(cell_head), "mil": cpu(mil)}, MODELS/"heads.pt")
json.dump({k: float(v) for k, v in resolved.items()}, open(MODELS/"thresholds.json", "w"), indent=2)
shutil.copy2(REPO/"src/aster_block2/decision_grid.yaml", MODELS/"decision_grid.yaml")
# quantifier.save() rounds TPR/FPR to 4 decimals for display; the kit needs the exact rates
json.dump({k: {"tpr": r.tpr, "fpr": r.fpr, "n_positive": r.n_positive, "n_negative": r.n_negative}
           for k, r in quantifier.rates.items()}, open(MODELS/"quantifier.json", "w"), indent=2)
domain_gate.save(MODELS/"ood_stats.npz")
json.dump({"bag_size": BAG, "seed": SEED, "image_size": 224, "trt_batch": TRT_BATCH,
           "embedding_dim": 512, "cell_classes": CELL_CLASSES, "gamma": thresholds.gamma,
           "run_id": run.manifest["run_id"], "torch_colab": torch.__version__,
           "onnx_sha256": hashlib.sha256((MODELS/"encoder.onnx").read_bytes()).hexdigest()},
          open(MODELS/"config.json", "w"), indent=2)

# 3. golden sessions: what the Jetson must reproduce, scored NOW by the same scorer
primary = test_frame.set_index("patient_id")
pool = test_frame[test_frame.n_classified >= 150].sort_values("n_classified")
picks = []
for label, k in (("acute_blastic__myeloid_oriented", 2), ("non_leukemic", 2), ("indeterminate", 1)):
    picks += pool[pool.label == label].patient_id.head(k).tolist()
picks += test_frame[test_frame.label == "out_of_domain"].sort_values("n_classified").patient_id.head(1).tolist()
expected = {}
for patient in picks:
    sources = [DATA/"corpus"/p for p in cai[cai.patient_id == patient].corpus_path.tolist()]
    target = GOLDEN/"sessions"/patient; target.mkdir()
    names = [f"{i:04d}_{p.name}" for i, p in enumerate(sources)]     # order = bag order
    for source, name in zip(sources, names): shutil.copy2(source, target/name)
    result, detail = score_session(sources, patient)
    assert result.label == primary.label[patient], f"{patient}: scorer no longer reproduces the primary run"
    expected[patient] = {"crops": names, "label": result.label, "tier": result.tier,
                         "n_classified": result.number_of_classified_leukocytes,
                         "counts": {k: int(v) for k, v in detail["counts"].items()},
                         "p_abn_raw": float(result.uncertainty["p_abn_raw"]),
                         "p_abn": float(result.uncertainty["p_abn"]), "ood": float(detail["ood"]),
                         "lineage_post": None if detail["lineage_post"] is None else float(detail["lineage_post"]),
                         "bag": [int(i) for i in deterministic_bag(len(sources), BAG, SEED)],
                         "verdicts": dict(result.verdicts)}
    print(f"golden {patient:<10} {len(sources):>4} crops  {result.label}")
json.dump(expected, open(GOLDEN/"expected.json", "w"), indent=2)

# 4. encoder parity: 16 preprocessed crops of the first golden session + fp32 outputs
first = picks[0]
transform = build_eval_transform(224)
tensors = torch.stack([transform(Image.open(GOLDEN/"sessions"/first/n).convert("RGB"))
                       for n in expected[first]["crops"][:16]])
with torch.inference_mode():
    feats = encoder(tensors.to(DEVICE))
    probs = cell_head(feats)["probabilities"]
np.savez_compressed(GOLDEN/"encoder_parity.npz", images=tensors.numpy(),
                    features=feats.cpu().numpy(), probabilities=probs.cpu().numpy())

# 5. checksums, verified on the Mac before the kit is assembled
files = sorted(p for p in KIT.rglob("*") if p.is_file())
(KIT/"COLAB_SHA256SUMS.txt").write_text("".join(
    f"{hashlib.sha256(p.read_bytes()).hexdigest()}  ./{p.relative_to(KIT)}\n" for p in files))
size = sum(p.stat().st_size for p in files) / 1e6
print(f"\njetson_kit: {len(files)} files, {size:.0f} MB  ->  {KIT}")
print("Download this folder from Drive, then on the Mac:  jetson/assemble_kit.sh <downloaded jetson_kit>")


## 13. Package for the prototype

Everything `integration/` needs, in one folder, hashed.

In [ ]:
# --- what leaves this notebook -----------------------------------------------
with run.section("13_bundle"):
    encoder.eval().cpu()
    run.checkpoint("cell_encoder", {"encoder": encoder.state_dict(),
                                    "cell_head": cell_head.state_dict()})
    run.checkpoint("mil_head_final", mil.state_dict())
    encoder.to(DEVICE)

    for path in (WORK/"checkpoints/encoder.onnx", WORK/"checkpoints/ood_stats.npz"):
        if path.exists(): run.bundle(path)
    run.bundle(run.root/"checkpoints/cell_encoder_final.pt", "cell_encoder.pt")
    run.bundle(run.root/"checkpoints/mil_head_final_final.pt", "mil_head.pt")
    run.bundle(REPO/"src/aster_block2/decision_grid.yaml")
    run.result("thresholds.json", {k: round(v, 6) for k, v in resolved.items()})
    run.bundle(run.root/"results/thresholds.json")
    run.bundle(REPO/"results/PREREGISTRATION.sha256")
    run.bundle(WORK/"checkpoints/quantifier.json")   # SessionScorer needs it
    run.bundle(DATA/"cai_inputs.pkl")                 # grid inputs, for any later replay
    run.result("bundle_manifest.json", {
        "cell_classes": CELL_CLASSES, "bag_size": BAG, "image_size": 224,
        "grid_sha256_at_freeze": GRID_SHA, "run_id": run.manifest["run_id"],
    })
    run.bundle(run.root/"results/bundle_manifest.json")

run.finalise(
    "Download runs/<run_id>/bundle/ and follow integration/PATCH.md. "
    "Build the TensorRT engine ON THE JETSON - engines are not portable. "
    "REPORT_INPUTS.md maps every artifact to the paper section it feeds."
)
print(open(run.root/"REPORT_INPUTS.md").read())

## 14. Morning summary\n\nRead this cell's output first after an unattended run.\n

In [ ]:

# --- morning summary: the one thing to read after an unattended run --------------------
from pathlib import Path
import json as _json

manifest = _json.loads((run.root/"manifest.json").read_text())
alerts = (run.root/"ALERTS.md")
print("=" * 72)
print(f"RUN {manifest['run_id']}   started {manifest['started'][:19]}")
print("=" * 72)

if alerts.exists():
    print(alerts.read_text())
else:
    print("\nNo alerts raised.\n")

for name in ("cell_head_training.json", "mil_head_training.json"):
    path = run.root/"results"/name
    if path.exists():
        print(f"{name}: {_json.loads(path.read_text())}")

path = run.root/"results/differential_spearman.json"
if path.exists():
    rho = _json.loads(path.read_text())
    print(f"\nSpearman rho, myeloblast: {rho.get('pb_myeloblast')}")

path = run.root/"results/caitomorph_409_predictions.csv"
if path.exists():
    import pandas as _pd
    frame = _pd.read_csv(path)
    print(f"\ncAItomorph: {len(frame)} patients")
    print(frame.label.value_counts().to_string())

path = run.root/"results/x40_stress.csv"
if path.exists():
    import pandas as _pd
    frame = _pd.read_csv(path)
    withheld = frame.label.isin(["out_of_domain","indeterminate","insufficient_evidence"]).mean()
    print(f"\nx40 stress: {len(frame)} sessions, {withheld:.0%} withheld "
          f"(old block 2 on the same material: AML 5/5, p=0.986)")

print(f"\nEverything is in {run.root}")
print("Read ALERTS.md first, then REPORT_INPUTS.md.")


## 15. Archive

Run last. Saves the executed notebook (every output), the kernel history, the environment,
the results printed but not saved, the AML-MLL feature cache and a sha256 inventory of the
whole run directory, then flushes Drive.


In [ ]:
# --- 15. Archive: everything this session produced, and how, into the Drive run directory ---
# One cell, run last. Each step is independent: a failure is reported, the others still run.
import json, shutil, subprocess, hashlib, time, sys, platform
from pathlib import Path
HIST = run.root/"history"; HIST.mkdir(exist_ok=True)
status = []
def step(name, fn):
    t0 = time.time()
    try:
        status.append((name, "OK", str(fn() or ""), time.time() - t0))
    except Exception as exc:
        status.append((name, "FAILED", f"{type(exc).__name__}: {exc}"[:200], time.time() - t0))

# 1. the notebook exactly as executed: every cell, every output, execution counts
def save_notebook():
    from google.colab import _message
    nb = _message.blocking_request("get_ipynb", request="", timeout_sec=300)["ipynb"]
    path = HIST/"ASTER_block2_executed.ipynb"
    path.write_text(json.dumps(nb, ensure_ascii=False, indent=1), encoding="utf-8")
    code = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
    return f"{len(nb.get('cells', []))} cells, {sum(1 for c in code if c.get('execution_count'))} executed, {path.stat().st_size/1e6:.1f} MB"
step("executed notebook (.ipynb)", save_notebook)

# 2. readable copies of it: HTML to browse, Markdown to quote in the report
def convert(fmt):
    def _run():
        out = subprocess.run([sys.executable, "-m", "nbconvert", "--to", fmt,
                              str(HIST/"ASTER_block2_executed.ipynb"), "--output-dir", str(HIST)],
                             capture_output=True, text=True)
        if out.returncode: raise RuntimeError(out.stderr.strip()[-200:])
        return fmt
    return _run
step("executed notebook (.html)", convert("html"))
step("executed notebook (.md)", convert("markdown"))

# 3. the kernel's own record: every submission since the kernel started, re-runs included
def kernel_history():
    ip = get_ipython()
    parts, count = [], 0
    for _session, number, (source, output) in ip.history_manager.get_range(session=0, raw=True, output=True):
        count += 1
        parts.append(f"# ===== execution [{number}] =====\n{source}\n")
        if output:
            parts.append(f"# ----- Out[{number}] -----\n# {str(output)[:2000]}\n")
    (HIST/"kernel_history.py").write_text("".join(parts), encoding="utf-8")
    hist_file = Path(str(ip.history_manager.hist_file))
    if hist_file.exists():
        shutil.copy2(hist_file, HIST/"ipython_history.sqlite")
    return f"{count} executions"
step("kernel execution history", kernel_history)

# 4. environment
def environment():
    (HIST/"pip_freeze.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                                      capture_output=True, text=True).stdout)
    try:
        (HIST/"nvidia_smi.txt").write_text(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    except FileNotFoundError:
        pass
    info = {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "run_id": run.manifest["run_id"],
            "archived_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    (HIST/"environment.json").write_text(json.dumps(info, indent=2))
    return info["gpu"]
step("environment", environment)

# 5. results that earlier cells printed without saving
def screening():
    from sklearn.metrics import roc_auc_score
    from statsmodels.stats.proportion import proportion_confint
    H, R, ACUTE = ["Stem cell donor"], ["Reactive changes"], ["AML", "ALL", "AL"]
    neoplasms = [d for d in test_frame.diagnosis_fine.unique() if d not in H + R]
    inside = test_frame[test_frame.label != "out_of_domain"]
    def auc(frame, pos, neg):
        s = frame[frame.diagnosis_fine.isin(pos + neg)]
        return float(roc_auc_score(s.diagnosis_fine.isin(pos).astype(int), s.p_abn)), int(len(s))
    rng = np.random.default_rng(0)       # same seed and call order as the reported cell
    def boot(pos, neg, B=2000):
        s = test_frame[test_frame.diagnosis_fine.isin(pos + neg)]
        y = s.diagnosis_fine.isin(pos).to_numpy().astype(int); p = s.p_abn.to_numpy()
        i_pos, i_neg = np.where(y == 1)[0], np.where(y == 0)[0]
        yb = np.r_[np.ones(len(i_pos)), np.zeros(len(i_neg))]
        v = [roc_auc_score(yb, np.r_[p[rng.choice(i_pos, len(i_pos))], p[rng.choice(i_neg, len(i_neg))]])
             for _ in range(B)]
        return [float(x) for x in np.percentile(v, [2.5, 97.5])]
    def wilson(k, n):
        lo, hi = proportion_confint(k, n, method="wilson")
        return {"k": int(k), "n": int(n), "rate": k / n, "ci95_wilson": [float(lo), float(hi)]}
    comparisons = {"AML vs donors": (["AML"], H), "acute vs donors": (ACUTE, H),
                   "acute vs donors+reactive": (ACUTE, H + R),
                   "any neoplasm vs donors+reactive": (neoplasms, H + R)}
    out = {"score": "calibrated P_abn, MIL head only (tier S screening)", "comparisons": {}}
    for name, (pos, neg) in comparisons.items():
        a, n = auc(test_frame, pos, neg); b, m = auc(inside, pos, neg)
        out["comparisons"][name] = {"auroc_all": a, "n_all": n, "auroc_excluding_ood": b, "n_excluding_ood": m}
    out["comparisons"]["AML vs donors"]["ci95_bootstrap_2000"] = boot(["AML"], H)
    out["comparisons"]["acute vs donors+reactive"]["ci95_bootstrap_2000"] = boot(ACUTE, H + R)
    aml = test_frame[test_frame.diagnosis_fine == "AML"]
    don = test_frame[test_frame.diagnosis_fine == "Stem cell donor"]
    out["operating_point"] = {"tau_calibrated": float(thresholds.p_abn),
                              "sensitivity_AML": wilson(int((aml.p_abn >= thresholds.p_abn).sum()), len(aml)),
                              "specificity_donors": wilson(int((don.p_abn < thresholds.p_abn).sum()), len(don))}
    out["old_block2_same_patients"] = {"auroc": 0.890, "sensitivity": wilson(25, 37),
                                       "specificity": wilson(99, 99),
                                       "source": "deployed block 2 (MAX head, 5 folds), 37 AML / 99 donors"}
    run.result("p_abn_screening.json", out)
    c = out["comparisons"]["AML vs donors"]
    return f"AML vs donors AUROC {c['auroc_all']:.3f} {[round(x, 3) for x in c['ci95_bootstrap_2000']]}"
step("P_abn screening results", screening)

def tables():
    run.result("caitomorph_409_confusion_full.csv",
               pd.crosstab(test_frame.diagnosis_fine, test_frame.label).reset_index())
    run.result("thresholds_final_state.json", dict(vars(thresholds)))
    return f"gamma {thresholds.gamma}"
step("confusion table + threshold state", tables)

# 6. the AML-MLL feature cache: later analyses without re-encoding
def feature_cache():
    source = DATA/"features_mll.pt"
    if not source.exists():
        return "not on local disk - skipped"
    (run.root/"cache").mkdir(exist_ok=True)
    shutil.copy2(source, run.root/"cache/features_mll.pt")
    return f"{source.stat().st_size/1e6:.0f} MB"
step("AML-MLL feature cache", feature_cache)

# 7. inventory of the whole run directory: path, size, sha256 - and where the weights are
def inventory():
    files = sorted(p for p in run.root.rglob("*") if p.is_file()
                   and p.name not in ("RUN_FILES.sha256", "RUN_INVENTORY.md"))
    with open(run.root/"RUN_FILES.sha256", "w") as handle:
        for p in files:
            handle.write(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  ./{p.relative_to(run.root)}\n")
    groups = {}
    for p in files:
        top = p.relative_to(run.root).parts[0] if len(p.relative_to(run.root).parts) > 1 else "(root)"
        groups.setdefault(top, [0, 0]); groups[top][0] += 1; groups[top][1] += p.stat().st_size
    weights = [p for p in files if p.suffix in (".pt", ".onnx", ".npz")]
    lines = [f"# Run {run.manifest['run_id']} — inventory\n", "| folder | files | MB |", "|---|---|---|"]
    lines += [f"| `{k}` | {n} | {s/1e6:.1f} |" for k, (n, s) in sorted(groups.items())]
    lines += ["", "## Model weights and statistics", "", "| file | MB |", "|---|---|"]
    lines += [f"| `{p.relative_to(run.root)}` | {p.stat().st_size/1e6:.1f} |" for p in weights]
    (run.root/"RUN_INVENTORY.md").write_text("\n".join(lines) + "\n")
    return f"{len(files)} files, {sum(s for _, s in groups.values())/1e6:.0f} MB, {len(weights)} weight/stat files"
step("inventory + sha256 of every file", inventory)

print(f"{'step':<36}{'status':<8}{'s':>6}  detail")
for name, state, detail, seconds in status:
    print(f"{name:<36}{state:<8}{seconds:>6.1f}  {detail}")
print(f"\nrun directory: {run.root}")
print(open(run.root/"RUN_INVENTORY.md").read() if (run.root/"RUN_INVENTORY.md").exists() else "")

# 8. make sure Drive has really received every byte before the session is closed
from google.colab import drive
drive.flush_and_unmount()
print("Drive flushed and unmounted - you can close this session and download the run folder.")
